[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week06_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 6 Assignment: Linearized Ratio Metrics

## QuickCart — Testing Revenue Per Search Session

QuickCart has launched a new search algorithm and wants to test whether it improves **revenue per search session** — the total revenue generated divided by the total number of search sessions.

This is a **ratio metric**: $\text{Revenue Per Session} = \frac{\sum \text{revenue}}{\sum \text{sessions}}$

The problem: users have vastly different numbers of sessions. A power user with 200 sessions contributes much more to the denominator than a casual user with 3 sessions. A naive per-user approach (compute ratio for each user, then t-test) is biased because it gives equal weight to all users regardless of their volume.

**The solution: Linearization.** Transform the ratio metric into a linear (per-user) metric that can be analyzed with standard methods:

$$L_i = Y_i - \kappa \cdot N_i$$

where:
- $Y_i$ = total revenue for user $i$
- $N_i$ = total sessions for user $i$
- $\kappa = \frac{\sum Y_i}{\sum N_i}$ (the global ratio, i.e., the metric value)

This linearized metric $L_i$ has the same mean comparison properties as the ratio but can be analyzed per-user with a t-test.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)

---

## Task 1: Implement Linearized Metric Calculation

### Context

The raw data has one row per event: `(user_id, date, revenue)`. Each row represents one search session, and the revenue is what was purchased in that session (0 if nothing was purchased).

The linearized metric for user $i$ is:

$$L_i = \sum_{\text{sessions}} \text{revenue}_i - \kappa \cdot \text{count\_sessions}_i$$

If `kappa` is not provided, compute it as the global ratio:
$$\kappa = \frac{\text{total revenue (all users)}}{\text{total sessions (all users)}}$$

### Problem

Implement `calculate_linearized_metric` that:
1. Filters data to the specified period (a tuple `(start_date, end_date)`)
2. Computes per-user: sum of revenue and count of sessions (rows)
3. If `kappa` is None, computes it as total_revenue / total_sessions
4. Computes the linearized metric: `revenue_per_user - kappa * sessions_per_user`
5. Returns a **tuple** `(DataFrame, kappa)`:
   - DataFrame has columns `[user_id_name, metric_name]`, filling 0 for missing users
   - `kappa` is the value used (computed or provided)

Returning kappa is important because you'll need it when combining linearization with CUPED across periods.

<details>
<summary>Hint 1 (click to expand)</summary>

Use groupby with two aggregations:
```python
user_stats = filtered_df.groupby(user_id_name).agg(
    total_value=(value_name, 'sum'),
    count=(value_name, 'count')  # number of sessions = number of rows
).reset_index()
```

</details>

<details>
<summary>Hint 2 (click to expand)</summary>

For users in `list_user_id` who have no sessions in the period, both their revenue and session count are 0, so their linearized metric is also 0. This is correct — they contribute no information about the ratio.

</details>

In [ ]:
def calculate_linearized_metric(
    df: pd.DataFrame,
    value_name: str,
    user_id_name: str,
    list_user_id: list,
    date_name: str,
    period: tuple,
    metric_name: str,
    kappa: float = None
) -> tuple:
    """Compute linearized ratio metric per user.
    
    The linearized metric transforms a ratio (sum_values / count_events)
    into a per-user metric suitable for t-tests:
        L_i = sum_values_i - kappa * count_events_i
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw event-level data. Each row is one event (e.g., search session).
    value_name : str
        Column with the numeric value (e.g., 'revenue').
    user_id_name : str
        Column with user identifier.
    list_user_id : list
        All user IDs that should appear in output.
    date_name : str
        Column with the date.
    period : tuple
        Tuple of (start_date, end_date) specifying date range (inclusive).
    metric_name : str
        Name for the output column.
    kappa : float, optional
        If None, computed as global total_value / total_count.
        If provided, used directly.
    
    Returns
    -------
    tuple of (pd.DataFrame, float)
        - DataFrame with columns [user_id_name, metric_name]
        - kappa value used (computed or provided)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test Task 1
test_df = pd.DataFrame({
    'user_id': ['A', 'A', 'A', 'B', 'B', 'C'],
    'date': pd.to_datetime(['2024-01-01'] * 6),
    'revenue': [10, 0, 5, 20, 0, 100]
})
# User A: 3 sessions, $15 revenue -> ratio = 5.0
# User B: 2 sessions, $20 revenue -> ratio = 10.0
# User C: 1 session, $100 revenue -> ratio = 100.0
# Global: 6 sessions, $135 revenue -> kappa = 22.5

all_users = ['A', 'B', 'C', 'D']
period = ('2024-01-01', '2024-01-31')  # tuple format

result, kappa_computed = calculate_linearized_metric(
    test_df, 'revenue', 'user_id', all_users, 'date', period, 'linearized'
)
print(f"Computed kappa: {kappa_computed:.2f}")
print(result)

# Verify kappa
assert abs(kappa_computed - 22.5) < 0.01, f"Expected kappa=22.5, got {kappa_computed}"

# Verify DataFrame
assert len(result) == 4, f"Expected 4 rows, got {len(result)}"
result_dict = result.set_index('user_id')['linearized'].to_dict()

assert abs(result_dict['A'] - (15 - 22.5 * 3)) < 0.01, f"User A: expected {15 - 22.5*3}, got {result_dict['A']}"
assert abs(result_dict['B'] - (20 - 22.5 * 2)) < 0.01, f"User B: expected {20 - 22.5*2}, got {result_dict['B']}"
assert abs(result_dict['C'] - (100 - 22.5 * 1)) < 0.01, f"User C: expected {100 - 22.5*1}, got {result_dict['C']}"
assert result_dict['D'] == 0, f"User D: expected 0, got {result_dict['D']}"

# Test with explicit kappa — function should return same kappa back
result_custom, kappa_back = calculate_linearized_metric(
    test_df, 'revenue', 'user_id', all_users, 'date', period, 'linearized', kappa=10.0
)
assert kappa_back == 10.0, f"When kappa provided, should return it back. Got {kappa_back}"
rd2 = result_custom.set_index('user_id')['linearized'].to_dict()
assert abs(rd2['A'] - (15 - 10 * 3)) < 0.01, "Custom kappa not applied correctly"

print("\nAll Task 1 checks passed!")

---

## Task 2: AA-Test Simulation — Comparing Three Approaches

### Context

Under the null hypothesis (no treatment effect), any valid test should produce **uniformly distributed p-values**. A test that doesn't is miscalibrated and will give you either too many false positives or too few true positives.

QuickCart's data scientists have been using a naive approach: compute revenue/sessions per user, then run a t-test. You suspect this is broken for ratio metrics.

### Problem

Run an AA-test simulation (no true effect) with 1000 iterations, comparing:

1. **Naive approach:** Compute `revenue / sessions` for each user, then two-sample t-test
2. **Sum-based approach:** Compare total revenue per user (ignoring session counts), t-test
3. **Linearized approach:** Use your `calculate_linearized_metric`, then two-sample t-test

For each approach, collect p-values and:
- Plot the p-value distribution (should be uniform under H0)
- Report the false positive rate at alpha=0.05 (should be ~5%)

**Data generation for each iteration:**
- 500 users per group
- Each user has `sessions ~ Poisson(10) + 1` sessions
- Each session generates revenue from `Exponential(mean=5)`
- No treatment effect (both groups are identical)

<details>
<summary>Hint 1: Why the naive approach fails</summary>

Users with few sessions have extremely noisy per-session averages. A user with 1 session and $50 revenue has a ratio of 50, while a user with 100 sessions averaging $5 each has a ratio of 5. The naive approach treats both estimates as equally reliable.

</details>

<details>
<summary>Hint 2: Simulation structure</summary>

```python
for i in range(n_simulations):
    # Generate data for control and treatment (identical distributions)
    # For each user in each group:
    #   n_sessions = np.random.poisson(10) + 1
    #   revenues = np.random.exponential(5, n_sessions)
    #   total_revenue = revenues.sum()
    #   ratio = total_revenue / n_sessions
    #   linearized = total_revenue - kappa * n_sessions
    
    # Compute kappa from ALL data (both groups combined)
    # kappa = sum(all_revenues) / sum(all_sessions)
    
    # Run t-tests for each approach
```

</details>

In [ ]:
# YOUR CODE HERE
#
# Suggested structure:
n_simulations = 1000
n_users_per_group = 500
alpha = 0.05

# Storage for p-values
pvalues_naive = []
pvalues_sum = []
pvalues_linearized = []

# Simulation loop
# YOUR CODE HERE
pass

In [ ]:
# Visualize p-value distributions
# A well-calibrated test produces a uniform (flat) histogram of p-values
#
# YOUR CODE HERE
# Suggested:
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for ax, pvals, title in zip(axes, 
#     [pvalues_naive, pvalues_sum, pvalues_linearized],
#     ['Naive (ratio/user)', 'Sum-based', 'Linearized']):
#     ax.hist(pvals, bins=20, edgecolor='black', density=True)
#     ax.axhline(1.0, color='red', linestyle='--', label='Uniform')
#     ax.set_title(title)
#     ax.set_xlabel('p-value')
#     ax.legend()
# plt.tight_layout()
# plt.show()
pass

In [ ]:
# Report false positive rates
# Uncomment after completing the simulation:

# fpr_naive = np.mean(np.array(pvalues_naive) < alpha)
# fpr_sum = np.mean(np.array(pvalues_sum) < alpha)
# fpr_linearized = np.mean(np.array(pvalues_linearized) < alpha)

# print(f"False Positive Rates (expected ~{alpha}):")
# print(f"  Naive (ratio per user):  {fpr_naive:.3f}")
# print(f"  Sum-based:               {fpr_sum:.3f}")
# print(f"  Linearized:              {fpr_linearized:.3f}")

# # The linearized approach should be well-calibrated
# assert 0.03 < fpr_linearized < 0.08, f"Linearized FPR {fpr_linearized:.3f} seems miscalibrated"
# print("\nLinearized approach is well-calibrated!")

---

## Task 3: Combining Linearization with CUPED

### Context

Linearization solves the ratio metric problem. CUPED reduces variance. Can we combine them?

**Yes!** The approach:
1. Compute the linearized metric for the **pre-pilot period** (using pre-period kappa)
2. Compute the linearized metric for the **pilot period** (using pilot-period kappa)
3. Apply CUPED: use the pre-period linearized metric as the covariate

### Problem

Implement the combined approach and demonstrate it on simulated data:

1. Generate data where users have stable session patterns (pre-period predicts pilot-period)
2. Compare four approaches on detecting a 3% improvement in revenue per session:
   - Naive ratio t-test
   - Linearized t-test
   - Linearized + CUPED
   - (Optional) Raw revenue CUPED without linearization

Run 500 simulations with a true 3% effect and report detection power for each.

<details>
<summary>Hint: Combining the two techniques</summary>

```python
# Step 1: Linearize for pre-period
kappa_pre = total_revenue_pre / total_sessions_pre
L_pre_i = revenue_pre_i - kappa_pre * sessions_pre_i

# Step 2: Linearize for pilot period
kappa_pilot = total_revenue_pilot / total_sessions_pilot
L_pilot_i = revenue_pilot_i - kappa_pilot * sessions_pilot_i

# Step 3: CUPED adjustment
theta = np.cov(L_pilot, L_pre)[0, 1] / np.var(L_pre)
L_cuped_i = L_pilot_i - theta * L_pre_i
```

</details>

<details>
<summary>Hint: Generating correlated session data</summary>

```python
# Each user has a stable "session rate" and "spend per session"
user_session_rate = np.random.gamma(3, 3, n_users)  # mean ~9 sessions
user_spend_rate = np.random.lognormal(1.5, 0.5, n_users)  # mean ~$5/session

# Pre-period and pilot-period data drawn from same user params
for user_i in range(n_users):
    n_sessions_pre = np.random.poisson(user_session_rate[user_i]) + 1
    n_sessions_pilot = np.random.poisson(user_session_rate[user_i]) + 1
    rev_pre = np.random.exponential(user_spend_rate[user_i], n_sessions_pre)
    rev_pilot = np.random.exponential(user_spend_rate[user_i], n_sessions_pilot)
```

</details>

In [ ]:
# YOUR CODE HERE
#
# Suggested structure:
# 1. Generate synthetic data with user-level random effects
# 2. For each of 500 simulations:
#    a. Generate pre-period and pilot-period data
#    b. Add 3% treatment effect to pilot revenue for treatment group
#    c. Compute metrics: naive ratio, linearized, linearized+CUPED
#    d. Run t-tests, collect p-values
# 3. Report power = fraction of p-values < 0.05

n_simulations = 500
n_users_per_group = 500
true_effect = 0.03  # 3% lift in revenue per session

# YOUR CODE HERE
pass

In [ ]:
# Report results
# Uncomment after completing:

# print("Detection Power (500 simulations, 3% true effect):")
# print(f"  Naive ratio:           {power_naive:.1%}")
# print(f"  Linearized:            {power_linearized:.1%}")
# print(f"  Linearized + CUPED:    {power_lin_cuped:.1%}")
# print()
# assert power_lin_cuped > power_linearized, "CUPED should improve power over linearized alone"
# assert power_linearized >= power_naive * 0.9, "Linearized should be at least as powerful as naive"
# print("Task 3 checks passed!")

---

## Bonus: Linearization and the Delta Method

### Context

The linearization approach is closely related to the **delta method** for ratio estimators. The delta method gives us the variance of a ratio:

$$\text{Var}\left(\frac{\bar{Y}}{\bar{N}}\right) \approx \frac{1}{\bar{N}^2} \left[ \text{Var}(Y_i) - 2\kappa \cdot \text{Cov}(Y_i, N_i) + \kappa^2 \cdot \text{Var}(N_i) \right]$$

Compare this to the variance of the linearized metric $L_i = Y_i - \kappa N_i$:

$$\text{Var}(L_i) = \text{Var}(Y_i) - 2\kappa \cdot \text{Cov}(Y_i, N_i) + \kappa^2 \cdot \text{Var}(N_i)$$

They differ only by the $\frac{1}{\bar{N}^2}$ scaling factor!

### Problem

1. Show empirically that `Var(L_i) / n` approximates `N_bar^2 * Var(ratio_hat_bootstrap)` where `ratio_hat_bootstrap` is the bootstrap distribution of the ratio estimator
2. Explain in 2-3 sentences: why does linearization give us a valid standard error for the ratio metric?

<details>
<summary>Hint</summary>

Bootstrap the ratio estimator 10,000 times:
```python
# Generate some data
n_users = 1000
sessions = np.random.poisson(10, n_users) + 1
revenues = np.array([np.random.exponential(5, s).sum() for s in sessions])

# Bootstrap
bootstrap_ratios = []
for _ in range(10000):
    idx = np.random.choice(n_users, n_users, replace=True)
    boot_ratio = revenues[idx].sum() / sessions[idx].sum()
    bootstrap_ratios.append(boot_ratio)
var_ratio_bootstrap = np.var(bootstrap_ratios)

# Compare to linearized variance
kappa = revenues.sum() / sessions.sum()
L = revenues - kappa * sessions
var_linearized_se = np.var(L) / (n_users * np.mean(sessions)**2)

print(f"Bootstrap Var(ratio): {var_ratio_bootstrap:.6f}")
print(f"Linearized estimate:  {var_linearized_se:.6f}")
```

</details>

In [ ]:
# YOUR CODE HERE (Bonus)
pass

**Your explanation:**

*Why does linearization give a valid standard error for the ratio metric? Write 2-3 sentences.*

---

## Summary

In this assignment you:
- Implemented linearized ratio metrics for proper A/B testing of ratios like revenue-per-session
- Demonstrated via simulation that naive per-user ratios produce miscalibrated tests
- Combined linearization with CUPED for maximum variance reduction
- (Bonus) Connected linearization to the delta method variance estimator

**Key takeaway:** Ratio metrics are ubiquitous (CTR, revenue per session, pages per visit). Always linearize them before applying standard hypothesis tests — the naive per-user ratio approach has incorrect type I error rates.